# STA272 — Introduction to Neural Networks

## Today: Neural Networks with Two Examples
- Two datasets:
  1) **CPS wages (HW6):** tabular prediction + evaluation + **ethics/fairness**
  2) **MNIST handwritten digits:** image classification where **neural nets** shine
- Goal: understand NNs, implement them, evaluate them, and discuss real-world implications.

## Learning objectives

- understand neural networks as a general model that has many applications in data science (e.g., LLMs, predicting protein structures, predicting housing prices or stock prices, predicting survival after a treatment, etc.)

- understand the architecture of a simple feed forward neural network with several layers

- implement a feedforward/multi layer perceptron neural network using `sklearn` in Python

- understand that a neural network is a logistic/linear regression without hidden layers

- understand how to evaluate a neural network

- understand potential ethical issues such as fairness and bias when using supervised machine learning models for predictions in the real world 

## Neural network models in data science

- Neural networks are used for many modern prediction tasks involving large data

- Data can consist of text, images, or numbers.  Text and images need to be transformed to numbers.

- The general workflow is:
    * Choose input data $X$ and output $y$
    * Train a neural network to minimize loss (training stage)
    * Evaluate the trained neural network on new data


## Example: converting text to data 

| word  | id | emb_1 | emb_2 |
|:------|---:|------:|------:|
| cats  | 0  | 0.9   | 0.1   |
| chase | 1  | 0.2   | 0.8   |
| mice  | 2  | 0.7   | 0.2   |
| love  | 3  | 0.1   | 0.9   |
| hate  | 4  | 0.1   | -0.9  |


- Words can be transformed to a pair of numbers (vectors) (how to get these numbers is a deeper question)
- Convert the sentence “cats chase mice” into numbers (vectors)
- Embeddings: `[[0.9, 0.1], [0.2, 0.8], [0.7, 0.2]]`
- This could be become input into a neural net where, for example, the model learns to place words with similar roles/meanings near each other (in vector space).

In [ ]:
import matplotlib.pyplot as plt

# Toy embeddings from the table above
embeddings = {
    "cats":  (0.9,  0.1),
    "chase": (0.2,  0.8),
    "mice":  (0.7,  0.2),
    "love":  (0.1,  0.9),
    "hate":  (0.1, -0.9),
}

# Extract x/y coordinates
words = list(embeddings.keys())
xs = [embeddings[w][0] for w in words]
ys = [embeddings[w][1] for w in words]

# Plot
plt.figure(figsize=(6, 6))
plt.scatter(xs, ys)

# Label each point
for w, x, y in zip(words, xs, ys):
    plt.text(x + 0.02, y + 0.02, w, fontsize=10)

# Helpful reference lines
plt.axhline(0, linewidth=1)
plt.axvline(0, linewidth=1)

plt.title("Toy 2D Word Embeddings")
plt.xlabel("Embedding dimension 1")
plt.ylabel("Embedding dimension 2")
plt.xlim(-1.1, 1.1)
plt.ylim(-1.1, 1.1)
plt.grid(True)
plt.show()

## Where does all this computing happen?

- Training neural networks — especially on large datasets — requires enormous computing power.
- We'll come back to the **real-world infrastructure** behind AI at the end of today's lecture.

## Feed-forward architecture vocabulary
- Dense layer, units, activation, output layer choices
- No hidden layers: linear/logistic regression
- Hidden layers allow nonlinear combinations

## Neural network - Multilayer Perceptron/Feedforward
![](dense_network_diagram.jpeg)



## Recurrent Neural Network 

- We will only study feed forward networks in this course.
  
![](rnn_diagram.jpeg)

## CPS Employment Data: Recent Graduates

Similar to the data used for decision trees, we use individual-level microdata from the [IPUMS CPS](https://cps.ipums.org/cps/) Annual Social and Economic Supplement (ASEC), 2020–2025.

The dataset has been filtered to **recent graduates**: ages 22–30 with a bachelor's degree or higher. 


- Predict `log_wage = log(INCWAGE)` from demographic/SES/occupation features

## Supervised Learning

- In **supervised learning**, we have a dataset with known inputs ($X$) and known outputs ($y$), and we train a model to predict $y$ from $X$.
- The model learns patterns from **training data**, and we evaluate how well it generalizes on **new (test) data** it hasn't seen before.
- All the models in this lecture — linear regression, logistic regression, and neural networks — are supervised learning methods.

We’ll use scikit-learn for MLP models.

In [ ]:
# Data
import numpy as np
import pandas as pd

# Split + scaling
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Models
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neural_network import MLPRegressor, MLPClassifier

# Metrics
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

## Load CPS data and create log_wage

Filter to employed with positive wages, then log-transform wages.

In [ ]:
# Load the CSV file (same dataset as HW6)
cps_df = pd.read_csv("cps_empl_new.csv")

# Keep only employed people with positive wages
cps_empl = cps_df[(cps_df["employed"] == 1) & (cps_df["INCWAGE"] > 0)].copy()

# Log-transform wages
cps_empl["log_wage"] = np.log(cps_empl["INCWAGE"])

cps_empl.shape

## One-hot encode race categories

Convert `race_cat` into numeric indicator columns using one-hot encoding.

One-hot encoding turns a categorical variable into multiple binary (0/1) columns—one column per category—where each observation has a 1 in the column for its category and 0s in all others.

In [ ]:
# Turn race_cat into indicator columns
cps_encoded = pd.get_dummies(cps_empl, columns=["race_cat"], drop_first=False)

# Show the race indicator columns
# use like parameter to keep variables like race_cat_
cps_encoded.filter(like="race_cat_").head()

In [ ]:
cps_encoded.shape

## Choose features and split train/test

Pick features aligned with HW6, then split into train/test.

In [ ]:
# Feature set aligned with HW6 answers
feature_cols = [
    "AGE", "female", "married", "educ_cat", "faminc_mid", "metro_binary",
    "us_citizen", "has_children", "insured", "OCC", "IND", "FIRMSIZE",
    "NCHILD", "YEAR",
    "race_cat_Asian", "race_cat_Black", "race_cat_Other", "race_cat_White"
]

X = cps_encoded[feature_cols].copy()
y = cps_encoded["log_wage"].copy()

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=272
)

## Scale features (important for neural nets)

Standardize features: fit on training data, transform both train and test.

- `scaler = StandardScaler()` creates an object that will compute per-feature **means** and **standard deviations**.  
- `X_train_s = scaler.fit_transform(X_train)` **fits** those statistics on the training data, then **transforms** `X_train` so each feature becomes $((x - \text{mean}) / \text{std})$.  
- `X_test_s = scaler.transform(X_test)` scales the test data using the **same** mean/std learned from the training set (avoids test-set information leakage).  


In [ ]:
# Standardize: mean 0, std 1 (fit on train only)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test) # applies mean/sd from training to test and avoids using data from training

## Baseline (regression): linear regression (no hidden layers)

- A feedforward neural network model with only input and output layers (i.e., no hidden layer) and a continuous output $y$ is equivalent to a linear regression model

Fit linear regression and evaluate RMSE and R² on the test set.

In [ ]:
lin = LinearRegression()
lin.fit(X_train_s, y_train)

pred_lin = lin.predict(X_test_s)
rmse_lin = np.sqrt(mean_squared_error(y_test, pred_lin))
r2_lin   = r2_score(y_test, pred_lin)

rmse_lin, r2_lin

print('Linear regression:  \n RMSE =', round(rmse_lin,2), '\n R-squared =', round(r2_lin,2))

## Neural net (regression): MLPRegressor with hidden layers

One hidden layer using ReLU. Use early stopping to reduce overfitting.

- `MLPRegressor` fits a **feed-forward neural network** for a **continuous outcome** (regression).  
- `hidden_layer_sizes=(32,)` means **one hidden layer** with **32 units**; more layers/units = more flexibility (and more overfitting risk).  
- `activation="relu"` adds **nonlinearity**, allowing the model to learn relationships that a purely linear model can’t.  
- `solver="adam"` is a gradient-based optimizer that updates the network’s weights to reduce prediction error.  
- `early_stopping=True` uses a **validation split** (`validation_fraction=0.2`) and stops training when validation performance stops improving (`n_iter_no_change=10`), helping reduce **overfitting**.  
- `rmse_mlp` and `r2_mlp` evaluate performance on the **test set**; `mlp_reg.n_iter_` tells you how many training iterations were actually used.  

In [ ]:
# Create a feed-forward neural network for regression
mlp_reg = MLPRegressor(
    hidden_layer_sizes=(32,),   # one hidden layer with 32 units (neurons)
    activation="relu",          # ReLU activation (adds nonlinearity)
    solver="adam",              # optimizer used to update weights
    max_iter=500,               # maximum training iterations (epochs)
    random_state=272,           # seed for reproducibility
    early_stopping=True,        # stop if validation performance stops improving
    validation_fraction=0.2,    # hold out 20% of training data for validation
    n_iter_no_change=10         # patience: stop after 10 iterations with no improvement
)

mlp_reg

### Fit the neural network on the training data

- `.fit()` is where the model actually learns — it adjusts the weights in the hidden layer to reduce prediction error on the training data.
- With `early_stopping=True`, the model monitors a validation set and stops when it stops improving.

In [ ]:
mlp_reg.fit(X_train_s, y_train)

print(f"Training stopped after {mlp_reg.n_iter_} iterations (out of max 500)")

### Predict and evaluate on the test set

- We use the trained model to predict on data it has **never seen** (the test set).
- RMSE tells us how far off the predictions are on average; R² tells us the fraction of variance explained.

In [ ]:
pred_mlp = mlp_reg.predict(X_test_s)

rmse_mlp = np.sqrt(mean_squared_error(y_test, pred_mlp))
r2_mlp   = r2_score(y_test, pred_mlp)

print(f"Neural network (MLP):\n  RMSE = {rmse_mlp:.2f}\n  R-squared = {r2_mlp:.2f}")

## What if we want to add more hidden layers?

- `hidden_layer_sizes=(32, 16)` means hidden layer 1 has 32 neurons and hidden layer 2 has 16 neurons.

- `hidden_layer_sizes=(32, 16, 15)` means hidden layer 1 has 32 neurons, hidden layer 2 has 16 neurons, and hidden layer 3 has 15 neurons.

- Adding more layers gives the model more capacity to learn "nonlinear" patterns.

- **How many layers and neurons should you choose?** There is no single rule — it is a **hyperparameter** you tune using a validation set (or cross-validation). A common practical approach: start with 1–2 hidden layers and a moderate number of neurons (e.g., 64 → 32), compare validation performance across a few candidate architectures, and stop when adding more capacity no longer improves validation error. More layers/neurons increase flexibility but also increase the risk of overfitting and slow down training.

## What's the neural network actually doing?

**Think of it like a committee making a decision:**

1. **Input layer:** You start with raw information about a person (age, education, occupation, etc.) — these are your features.

2. **Hidden layer — the "committee members":** Each neuron is like a committee member who looks at *all* the inputs, weighs them differently based on what they've learned is important, and forms their own opinion (a single number). Some members might focus on education + occupation, others on age + location — the network figures out what combinations are useful.

3. **The activation "filter" (ReLU):** After each committee member forms their opinion, they apply a simple rule: *if my opinion is negative, I'll stay quiet (output 0); if it's positive, I'll speak up.* This is what ReLU does — it filters out negative signals and lets positive ones pass through.

4. **Output layer — the "final vote":** The output layer takes all the committee members' filtered opinions, combines them with another set of weights, and produces a final prediction (a number for regression, a probability for classification).

5. **Training — learning the weights:** The network starts with random weights and gradually adjusts them to reduce prediction error. This is like the committee learning from examples: "we predicted too high for this person, let's adjust."

**Key insight:** With no hidden layer (no committee), you just have a direct weighted sum of inputs — that's exactly linear/logistic regression. The hidden layers let the model learn **combinations of features** that a straight line can't capture.

## Early stopping (what it is + why it helps)
- Uses a validation split and stops training when validation performance stops improving
- Helps reduce overfitting (prevents training "too long")
- Often saves training time
- **Other ways to reduce overfitting:** use more training data; add regularization (e.g., L2 penalty via `alpha` in sklearn); use dropout (randomly disabling neurons during training); reduce model complexity (fewer layers/neurons)

## Binary outcome: “high wage” above median

Create a binary label and scale features for classification.

In [ ]:
# Compute the median wage in the dataset (used as a threshold)
median_wage = cps_encoded["INCWAGE"].median()

print(median_wage)

# Create a binary label: 1 if wage is above the median, 0 otherwise
cps_encoded["high_wage"] = (cps_encoded["INCWAGE"] > median_wage).astype(int)

# Target variable for binary classification
y_bin = cps_encoded["high_wage"]

# sanity check
y_bin.value_counts()/sum(y_bin.value_counts())

In [ ]:
# Split into training and test sets
# - test_size=0.20 means 20% test, 80% train
# - stratify=y_bin keeps the 0/1 class balance similar in train and test
Xb_train, Xb_test, yb_train, yb_test = train_test_split(
    X, y_bin, test_size=0.20, random_state=272, stratify=y_bin
)

# Standardize features (fit scaler on training data only)
scaler_b = StandardScaler()
Xb_train_s = scaler_b.fit_transform(Xb_train)  # learn mean/std from train, then scale train
Xb_test_s  = scaler_b.transform(Xb_test)       # scale test using the train mean/std

## Binary baseline: logistic regression (no hidden layers)

Fit logistic regression and compute accuracy + confusion matrix.

- **Seaborn (`sns`)** is a Python visualization library built on top of matplotlib that makes statistical plots (like heatmaps) easier to create and style — here we use `sns.heatmap()` to display the confusion matrix with colour-coded counts.

In [ ]:
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    precision_score, recall_score, f1_score
)

logit = LogisticRegression(max_iter=2000, solver="lbfgs")
logit.fit(Xb_train_s, yb_train)

pred_logit = logit.predict(Xb_test_s)

# Summary metrics
acc_logit  = accuracy_score(yb_test, pred_logit)
prec_logit = precision_score(yb_test, pred_logit)
rec_logit  = recall_score(yb_test, pred_logit)
f1_logit   = f1_score(yb_test, pred_logit)

print(f"Logistic Regression — Accuracy: {acc_logit:.2f}")

# Confusion matrix heatmap
cm_logit = confusion_matrix(yb_test, pred_logit)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm_logit, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Low wage", "High wage"],
            yticklabels=["Low wage", "High wage"], ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Logistic Regression — Confusion Matrix")
plt.tight_layout()
plt.show()

## Binary neural net: MLPClassifier

Same idea as regression MLP, but for classification.

In [ ]:
mlp_bin = MLPClassifier(
    hidden_layer_sizes=(32,),
    activation="relu",
    solver="adam",
    max_iter=500,
    random_state=272,
    early_stopping=True,
    validation_fraction=0.2,
    n_iter_no_change=10
)

mlp_bin.fit(Xb_train_s, yb_train)
pred_mlp_bin = mlp_bin.predict(Xb_test_s)

# Summary metrics
acc_mlp  = accuracy_score(yb_test, pred_mlp_bin)
prec_mlp = precision_score(yb_test, pred_mlp_bin)
rec_mlp  = recall_score(yb_test, pred_mlp_bin)
f1_mlp   = f1_score(yb_test, pred_mlp_bin)

print(f"MLPClassifier — Accuracy: {acc_mlp:.2f}  (trained {mlp_bin.n_iter_} iterations)")

# Confusion matrix heatmap
cm_mlp = confusion_matrix(yb_test, pred_mlp_bin)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm_mlp, annot=True, fmt="d", cmap="Oranges",
            xticklabels=["Low wage", "High wage"],
            yticklabels=["Low wage", "High wage"], ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Neural Network (MLP) — Confusion Matrix")
plt.tight_layout()
plt.show()

## Comparing Logistic Regression and Neural Network (Binary)

- **Takeaway:** The neural network gives a **small improvement** (70% → 71% accuracy), mainly by reducing false positives.
- For this tabular dataset, the gain from adding hidden layers is modest.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

labels = ['Accuracy', 'Precision', 'Recall', 'F1']
logit_scores = [acc_logit, prec_logit, rec_logit, f1_logit]
mlp_scores   = [acc_mlp, prec_mlp, rec_mlp, f1_mlp]

x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 4))
bars1 = ax.bar(x - width/2, logit_scores, width, label='Logistic Regression', color='steelblue')
bars2 = ax.bar(x + width/2, mlp_scores, width, label='Neural Network (MLP)', color='coral')

ax.set_ylabel('Score')
ax.set_title('Binary Classification: Logistic Regression vs Neural Network')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylim(0.5, 0.8)
ax.legend()
ax.bar_label(bars1, fmt='%.2f', fontsize=9)
ax.bar_label(bars2, fmt='%.2f', fontsize=9)
plt.tight_layout()
plt.show()

## Multiclass outcome: wage quartiles (4 classes)

Create 4 wage classes using quartiles; scale features.

In [ ]:
cps_encoded["wage_q"] = pd.qcut(cps_encoded["INCWAGE"], q=4, labels=False)
y_multi = cps_encoded["wage_q"].astype(int)

Xm_train, Xm_test, ym_train, ym_test = train_test_split(
    X, y_multi, test_size=0.20, random_state=272, stratify=y_multi
)

scaler_m = StandardScaler()
Xm_train_s = scaler_m.fit_transform(Xm_train)
Xm_test_s  = scaler_m.transform(Xm_test)

## Multiclass baseline: multinomial logistic regression

Softmax regression baseline; evaluate accuracy + confusion matrix.

- **Softmax regression** is the multiclass extension of logistic regression: instead of outputting a single probability for one class, it outputs a probability for *each* class (all summing to 1) and predicts whichever class has the highest probability.

In [ ]:
softmax = LogisticRegression(max_iter=4000, solver="lbfgs")
softmax.fit(Xm_train_s, ym_train)

pred_softmax = softmax.predict(Xm_test_s)
acc_softmax = accuracy_score(ym_test, pred_softmax)

print(f"Multiclass Logistic Regression — Accuracy: {acc_softmax:.2f}")

# Confusion matrix heatmap
cm_softmax = confusion_matrix(ym_test, pred_softmax)
q_labels = ["Q1 (lowest)", "Q2", "Q3", "Q4 (highest)"]
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm_softmax, annot=True, fmt="d", cmap="Blues",
            xticklabels=q_labels, yticklabels=q_labels, ax=ax)
ax.set_xlabel("Predicted quartile")
ax.set_ylabel("Actual quartile")
ax.set_title("Logistic Regression — Confusion Matrix (4 wage quartiles)")
plt.tight_layout()
plt.show()

## Multiclass neural net: MLPClassifier

MLPClassifier with hidden layers; compare to multinomial logistic.

In [ ]:
mlp_multi = MLPClassifier(
    hidden_layer_sizes=(32, 16),
    activation="relu",
    solver="adam",
    max_iter=500,
    random_state=272,
    early_stopping=True,
    validation_fraction=0.2,
    n_iter_no_change=10
)

mlp_multi.fit(Xm_train_s, ym_train)
pred_mlp_multi = mlp_multi.predict(Xm_test_s)
acc_mlp_multi = accuracy_score(ym_test, pred_mlp_multi)

print(f"MLPClassifier — Accuracy: {acc_mlp_multi:.2f}  (trained {mlp_multi.n_iter_} iterations)")

# Confusion matrix heatmap
cm_mlp_multi = confusion_matrix(ym_test, pred_mlp_multi)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm_mlp_multi, annot=True, fmt="d", cmap="Oranges",
            xticklabels=q_labels, yticklabels=q_labels, ax=ax)
ax.set_xlabel("Predicted quartile")
ax.set_ylabel("Actual quartile")
ax.set_title("Neural Network (MLP) — Confusion Matrix (4 wage quartiles)")
plt.tight_layout()
plt.show()

## Comparing Multiclass Models

- Both models struggle with the **middle quartiles** (classes 1 and 2) — nearby wage groups are hard to tell apart.
- Both do best at the **extremes** (lowest and highest wage quartiles).
- The neural network gives a **very small improvement** overall.
- **Takeaway:** For this tabular data, adding hidden layers doesn't help much — the features don't have the complex nonlinear patterns that neural nets excel at finding.

In [ ]:
from sklearn.metrics import f1_score

labels = ['Q1\n(lowest)', 'Q2', 'Q3', 'Q4\n(highest)']
softmax_f1 = f1_score(ym_test, pred_softmax, average=None)
mlp_f1     = f1_score(ym_test, pred_mlp_multi, average=None)

x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 4))
bars1 = ax.bar(x - width/2, softmax_f1, width, label='Logistic Regression', color='steelblue')
bars2 = ax.bar(x + width/2, mlp_f1, width, label='Neural Network (MLP)', color='coral')

ax.set_ylabel('F1 Score')
ax.set_title('Multiclass: F1 Score by Wage Quartile')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylim(0, 0.7)
ax.legend()
ax.bar_label(bars1, fmt='%.2f', fontsize=9)
ax.bar_label(bars2, fmt='%.2f', fontsize=9)
plt.tight_layout()
plt.show()

## Ethical Considerations

- **Discrimination risk:** Predictions can be used to justify unequal treatment (e.g., hiring, pay offers, lending) based on protected attributes like race or sex.  

- **Amplifying historical bias:** If past outcomes reflect discrimination or unequal opportunity, the model can **learn and reproduce** those patterns.  

- **Proxy effects and "fairness washing":** Even if you remove race/sex later, other variables (ZIP code, occupation, school, etc.) can act as proxies; including race/sex without a clear fairness plan can also be used to claim the model is "fair" without evidence.  

- **Unequal errors across groups:** Models may have higher false positives/false negatives for some groups, leading to uneven harms (e.g., systematically underpredicting wages for one group).  

- **Stigmatization and trust:** Using protected attributes can feel invasive or stigmatizing, reduce trust, and create reputational/legal risk if people view the system as unfair.  

### Discussion scenario

> Imagine a company uses our wage-prediction model to set **starting salary offers** for new graduates. We saw that the model's RMSE is higher for Black and Asian graduates than for White graduates. What could go wrong?

- The model might **systematically lowball offers** for some groups if it underpredicts their wages.
- Even if the *average* offer is "fair," the **wider error spread** for certain groups means more individuals get unreasonably low (or high) offers.
- The company might not even notice — the overall RMSE looks fine. **Who is checking group-level accuracy?**
- *What safeguards would you want before deploying a model like this?*

## Subgroup evaluation: RMSE by sex group (regression)

Compute regression error separately by `female`.

In [ ]:
test_results = X_test.copy()
test_results["y_true"] = y_test.values
test_results["y_pred"] = pred_mlp

for g, sub in test_results.groupby("female"):
    rmse_g = np.sqrt(mean_squared_error(sub["y_true"], sub["y_pred"]))
    print("female =", g, "RMSE =", rmse_g)

## Subgroup evaluation: RMSE by race group (regression)

Label race group from one-hot columns; compute RMSE by group.

In [ ]:
race_cols = ["race_cat_Asian", "race_cat_Black", "race_cat_Other", "race_cat_White"]
race_labels = ["Asian", "Black", "Other", "White"]

race_idx = test_results[race_cols].values.argmax(axis=1)
test_results["race_group"] = [race_labels[i] for i in race_idx]

for g, sub in test_results.groupby("race_group"):
    rmse_g = np.sqrt(mean_squared_error(sub["y_true"], sub["y_pred"]))
    print("race =", g, "RMSE =", rmse_g)

## Is the model biased/unfair?

- **Unequal error rates across groups:** RMSE is higher for women than men (0.763 vs 0.752) and much higher for Asian/Black groups than for White (≈0.84 vs 0.73), meaning the model is **less accurate for some groups**.  
- **Potential harm if used for decisions:** If predictions inform pay offers, hiring, promotions, or counseling, higher error for certain groups can lead to **systematically worse decisions** for them.  
- **Fairness concern beyond “average performance”:** Even if overall RMSE looks acceptable, these results show the model does **not generalize equally**—ethical evaluation should include **group-level metrics**, not just overall RMSE.  
- **Possible reasons (warning sign, not proof of bias):** Differences could come from unequal sample sizes/data quality, different wage distributions, missing predictors that matter more for some groups, or structural inequities reflected in the data.  

## MNIST Data

- **MNIST** (Modified National Institute of Standards and Technology) is a dataset of handwritten digit images (0–9) widely used as a benchmark for image classification.
- Each image is a 28×28 grayscale grid of pixel intensities, which gets **flattened** into a vector of 784 numbers — this becomes the input to a neural network.
- The task is **multiclass classification**: given the 784 pixel values, predict which digit (0–9) the image represents.
- MNIST is a natural example of where neural networks shine — the relationship between raw pixels and digit identity is highly nonlinear, and neural nets can learn useful intermediate representations (edges, curves, loops) in their hidden layers.
- We will compare logistic regression (no hidden layers) to an MLP classifier on this task.

## A brief history of MNIST

- Created in **1998** by Yann LeCun, Corinna Cortes, and Christopher Burges by remixing handwritten digit samples from US Census Bureau employees and high school students.
- Originally used to evaluate LeCun's **LeNet-5** convolutional neural network — one of the earliest demonstrations that neural networks could achieve near-human accuracy on image recognition.
- Became the standard "hello world" benchmark for machine learning and computer vision; nearly every new classification method has been tested on it.
- Contains **70,000 images** (60,000 training, 10,000 test), each a 28×28 grayscale image of a single handwritten digit.
- State-of-the-art models now achieve **>99.8%** accuracy on MNIST — the dataset is largely considered "solved," but it remains an excellent teaching tool for understanding how neural networks process image data.

## How an image becomes data

- A grayscale image is stored as a **grid of pixels**, where each pixel holds a single number representing its brightness (0 = black, 255 = white).
- For MNIST, each image is a **28 × 28 grid**, so there are **784 pixels** in total.
- To feed the image into a neural network, we **flatten** the 2D grid into a **1D vector** of length 784 — each pixel becomes one input feature.

$$
\underbrace{\begin{bmatrix} 0 & 0 & 150 & \cdots \\ 0 & 200 & 255 & \cdots \\ \vdots & \vdots & \vdots & \ddots \end{bmatrix}}_{28 \times 28 \text{ image}}
\;\longrightarrow\;
\underbrace{(0,\; 0,\; 150,\; \ldots,\; 0,\; 200,\; 255,\; \ldots)}_{784 \text{ features}}
$$

- Pixel values are typically **scaled** (e.g., divided by 255 to get values in [0, 1], or standardized to mean 0 / std 1) before training — this helps gradient-based optimizers converge faster.
- After flattening, the image is just a **row of 784 numbers**, no different from any other tabular dataset with 784 columns — the neural network treats it the same way it treated the CPS wage features above.

## What does the model actually "see"?

- To us, an image of a handwritten digit looks like a shape. To the model, it's just a grid of numbers.
- Below we show the **same image** two ways: as a picture (left) and as the raw pixel values the model receives (right).
- Brighter squares = higher pixel values = more "ink" on the paper.

In [ ]:
from sklearn.datasets import load_digits
import matplotlib.pyplot as plt
import numpy as np

# Use the small 8x8 digits for a clear demo (numbers fit in each cell)
digits = load_digits()
sample_img = digits.images[0]  # an 8x8 image of a "0"

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Left: the image as we see it
axes[0].imshow(sample_img, cmap="gray")
axes[0].set_title(f"What we see (digit: {digits.target[0]})", fontsize=13)
axes[0].axis("off")

# Right: the raw pixel values as a heatmap with numbers
im = axes[1].imshow(sample_img, cmap="YlOrRd")
axes[1].set_title("What the model sees (pixel values)", fontsize=13)
for i in range(8):
    for j in range(8):
        val = int(sample_img[i, j])
        axes[1].text(j, i, str(val), ha="center", va="center",
                     fontsize=9, color="black" if val < 10 else "white")
axes[1].set_xticks(range(8))
axes[1].set_yticks(range(8))
axes[1].set_xlabel("Pixel column")
axes[1].set_ylabel("Pixel row")

plt.tight_layout()
plt.show()

## Warm-up: sklearn's built-in 8×8 digits dataset

- Before using the full 28×28 MNIST, we start with a **smaller version** bundled with scikit-learn: 1,797 images of digits, each only **8×8 pixels** (64 features instead of 784).
- This runs fast and lets us see the full workflow before scaling up.

### Load the 8×8 digits data

- `load_digits()` returns a small dataset of handwritten digits already included in scikit-learn.
- `digits.data` is the flattened version (each row = 64 pixel values); `digits.images` keeps the 8×8 grid for plotting.

In [ ]:
from sklearn.datasets import load_digits
import matplotlib.pyplot as plt

digits = load_digits()
X = digits.data        # shape: (1797, 64) — flattened 8x8 images
y = digits.target      # shape: (1797,)    — labels 0..9
images = digits.images # shape: (1797, 8, 8) — original grids

print("X shape:", X.shape)
print("y shape:", y.shape)

### Split into training and test sets

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

### Scale features and train the MLP

- We use `make_pipeline` to chain scaling and the MLP into a single object — this keeps the code short and ensures scaling is always applied consistently.
- Two hidden layers with 64 and 32 units.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neural_network import MLPClassifier

model = make_pipeline(
    StandardScaler(),
    MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=0)
)

model.fit(X_train, y_train)

### Evaluate accuracy and confusion matrix

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix

y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

### Visualize a few example images with predictions

- Each 8×8 image is shown alongside the model's predicted label and the true label.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for ax, idx in zip(axes.ravel(), range(10)):
    ax.imshow(images[idx], cmap="gray")
    ax.set_title(f"true={y[idx]}, pred={model.predict(X[idx:idx+1])[0]}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## Full MNIST (28×28 images, 70,000 samples)

- Now we use the real MNIST dataset: **70,000 images**, each **28×28 pixels** (784 features).
- This is a much larger and more realistic dataset than the 8×8 warm-up above.

### Load MNIST and scale pixel values

- `fetch_openml` downloads the full MNIST dataset (first run caches it locally).
- We divide pixel values by 255 so they fall in [0, 1] — this is a simple form of scaling.

In [ ]:
from sklearn.datasets import fetch_openml

mnist = fetch_openml("mnist_784", version=1, as_frame=False)

X = mnist.data.astype("float32")   # (70000, 784)
y = mnist.target.astype("int64")   # labels 0..9

# Scale pixels to [0, 1]
X = X / 255.0

print("X shape:", X.shape)
print("y shape:", y.shape)

### Train/test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

### Train an MLP classifier on full MNIST

- Two hidden layers with 128 and 64 units.
- `max_iter=20` limits training to 20 epochs — MNIST is large, so even a few epochs can give good results.
- Scaling via `StandardScaler` inside the pipeline further standardizes the already-scaled pixel values.

In [ ]:
model = make_pipeline(
    StandardScaler(),
    MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=20, random_state=0)
)

model.fit(X_train, y_train)

### Evaluate the MLP on the test set

In [ ]:
y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

### Visualize a few MNIST images with predictions

- We reshape each 784-length row back to 28×28 for display.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for ax, idx in zip(axes.ravel(), range(10)):
    img = X[idx].reshape(28, 28)
    ax.imshow(img, cmap="gray")
    ax.set_title(f"true={y[idx]}, pred={model.predict(X[idx:idx+1])[0]}")
    ax.axis("off")
plt.tight_layout()
plt.show()

### Try it yourself!

- Change the value of `idx` below to any number between 0 and 13,999 to see a different test image.
- Can you find one the model gets **wrong**?

In [ ]:
# Change this number to explore different test images!
idx = 42

img = X_test[idx].reshape(28, 28)
true_label = y_test[idx]
pred_label = model.predict(X_test[idx:idx+1])[0]

plt.figure(figsize=(4, 4))
plt.imshow(img, cmap="gray")
plt.title(f"True: {true_label}    Predicted: {pred_label}",
          fontsize=14, color="green" if true_label == pred_label else "red")
plt.axis("off")
plt.show()

### Baseline: logistic regression (no hidden layers) on MNIST

- Just like with the CPS data, we compare the neural network to a **logistic regression** baseline (equivalent to a neural network with no hidden layers).
- The `saga` solver is efficient for large datasets; `tol=0.1` allows it to converge faster with a looser tolerance.

In [ ]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(solver='saga', max_iter=100, tol=0.1)
clf.fit(X_train, y_train)

accuracy = clf.score(X_test, y_test)
print(f"Logistic Regression Test Accuracy: {accuracy * 100:.2f}%")

### Visualize logistic regression predictions

- Let's look at 10 test images and compare the logistic regression predictions to the true labels.

In [ ]:
samples = X_test[:10]
predictions = clf.predict(samples)
actual_labels = y_test[:10]

plt.figure(figsize=(12, 5))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    image = samples[i].reshape(28, 28)
    plt.imshow(image, cmap='gray')
    plt.title(f"Pred: {predictions[i]}\nActual: {actual_labels[i]}")
    plt.axis('off')
plt.tight_layout()
plt.show()

### Comparing MLP and Logistic Regression on MNIST

- **Logistic regression** (no hidden layers): ~**92.6%** accuracy
- **MLP** (two hidden layers: 128, 64 units): ~**97.5%** accuracy
- The neural network reduces the error rate by more than **half** (7.4% → 2.5%).
- Unlike the CPS wage data — where the MLP gave only a small improvement over logistic regression — on MNIST the hidden layers make a **large difference**, because the relationship between raw pixels and digit identity is highly nonlinear.
- **Takeaway:** neural networks shine when the features (pixels) need complex, nonlinear combinations to produce good predictions. For simpler tabular data the gain over logistic regression can be modest.

## The bigger picture: AI data centres

Training these models at scale requires enormous compute — here's what that looks like in the real world.

- **AI scaling laws** are empirical rules showing that model performance improves predictably as you increase training compute, data, and model size (often following a smooth power-law trend).
- This has incentivized building more compute for AI models (bigger models/data/training)
- Data centres support **training** (weight updates) and **inference** (serving predictions)
- AI workloads need massive compute (GPU clusters)
- Data centres raise issues: electricity demand, cooling/water, local siting

[![](Guardian_datacentre.png)](https://www.theguardian.com/technology/2026/mar/01/datacentre-developers-energy-greenhouse-gas-emissions)

## What we learned today

1. **A neural network is regression + hidden layers.** Remove the hidden layers and you get linear or logistic regression.

2. **Hidden layers learn useful combinations of features.** Each neuron combines inputs in a different way, letting the model capture nonlinear patterns.

3. **Neural nets don't always win.** On the CPS tabular data, the MLP barely beat logistic regression. On MNIST images, it cut the error rate by more than half. The benefit depends on **how complex the patterns in the data are.**

4. **The workflow is the same every time:** load data, split train/test, scale features, fit the model, evaluate on the test set.

5. **Fairness matters.** A model can look good overall but perform **much worse for certain groups.** Always check group-level metrics before deploying a model for real decisions.

6. **Training at scale has real-world costs** — energy, water, infrastructure — that are important to consider as AI grows.